# To perform and visualize analyses online 

## Before the experiment: get everything ready 

### imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from omegaconf import DictConfig, OmegaConf
import hydra
from hydra.utils import instantiate
from hydra.core.config_store import ConfigStore
from hydra import compose, initialize
from IPython.display import display
import sys
import os
from pathlib import Path


### SET THIS PATH FROM THE CURRENT WORKING DIRECTORY TO THE REPO DIRECTORY
relative_repo_path = "GitRepos/simulation_closed_loop"

In [3]:
# Add the parent directory to the path so we can import modules properly
cwd = Path.cwd()
full_repo_path = os.path.join(cwd, relative_repo_path)

if not os.path.exists(full_repo_path):
    raise ValueError(f"The specified relative path to the repo does not exist: {full_repo_path}\
                     Check workding dir and relative repo dir path variable.")

print(f"Working directoty: {cwd}")
print(f"Full repo path: {full_repo_path}")

# append repo path 
sys.path.append(full_repo_path)



# Initialize Hydra with the relative path to the config directory
config_path = os.path.join(relative_repo_path,"model_in_the_loop/config")
print(f"Config path relative to cwd: {config_path}")
if not os.path.exists(os.path.join(cwd,config_path)):
    raise ValueError(f"The specified config path does not exist: {os.path.join(cwd,config_path)}\
                     Check workding dir and config dir path variable.")

# Initialize Hydra
with initialize(version_base="1.3", config_path=config_path):
    # Compose the configuration
    cfg = compose(config_name="config")

# Print the config to verify it loaded correctly
print("Configuration loaded successfully:")



Working directoty: /gpfs01/euler/User/ssuhai
Full repo path: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop
Config path relative to cwd: GitRepos/simulation_closed_loop/model_in_the_loop/config


Configuration loaded successfully:


In [4]:
from model_in_the_loop.core import (DJTableHolder,Preprocessor,QualityAndTypeWrapper,STAWrapper,RandomSeedMEIWrapper,
                                                     )

from model_in_the_loop.utils.plotting import show_all_rois_plot
from model_in_the_loop.utils.file_management import copy_rec_files,create_directory_structure

from model_in_the_loop.utils.transform_to_avi_stimulus import create_single_mei_avis_and_metadata
from model_in_the_loop.utils.simple_logging import log

### Create processing components (connect them to DB)

In [6]:
# create preprocessor
os.environ["DJ_SUPPORT_FILEPATH_MANAGEMENT"] = "TRUE"

dj_table_holder = DJTableHolder(
                username=cfg.DJ.username, # type: ignore
                
                #paths
                home_directory=cfg.paths.home_directory, # type: ignore
                repo_directory=cfg.paths.repo_directory, # type: ignore
                dj_config_directory= cfg.paths.dj_config_directory, # type: ignore
                rgc_output_directory= cfg.paths.rgc_output_directory, # type: ignore


                userinfo= cfg.DJ.userinfo, # type: ignore

                table_parameters=cfg.DJ.table_parameters, # type: ignore

                # from overall configs
                debug=cfg.debug, # type: ignore
                plot_results=cfg.plot_results, # type: ignore
                )



In [7]:
# # # Load config and tables
dj_table_holder.setup()

[2025-10-18 21:13:05,187][INFO]: Connecting ssuhai@172.25.240.200:3306


[2025-10-18 21:13:05,246][INFO]: Connected ssuhai@172.25.240.200:3306


schema_name: ageuler_ssuhai_closed_loop
Done reconnecting. Skipping adding new entries from config.


/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/core/dj_wrappers/base.py:291: UserWarning: 
Some DJ tables (like UserInfo) are not empty, skipping adding new entries from config.
Make sure this is wanted. Call clear_tables(`all`) if you want different data in there
  warnings.warn("\nSome DJ tables (like UserInfo) are not empty, skipping adding new entries from config.\nMake sure this is wanted. Call clear_tables(`all`) if you want different data in there")


In [8]:
preprocessor = Preprocessor(dj_table_holder=dj_table_holder)


quality_type_analysis_wrapper = QualityAndTypeWrapper(
    dj_table_holder=dj_table_holder,)

sta_wrapper = STAWrapper(
    dj_table_holder=dj_table_holder,)

random_seed_mei_wrapper = RandomSeedMEIWrapper(
    dj_table_holder=dj_table_holder,
    cfg=cfg,
    seeds=[111,222]
)

## During the experimet

### Move files from server to the repo 

In [ ]:
create_directory_structure(base_directory= cfg.DJ.userinfo.data_dir,)


copy_rec_files(
    recording_files_dir=cfg.paths.recording_files_dir,  # type: ignore
    destination_base=cfg.DJ.userinfo.data_dir,  # type: ignore
    full_dummy_ini_dir= os.path.join(cfg.paths.repo_directory, cfg.paths.dummy_ini_dir),  # type: ignore
)

### Preprocessing

In [9]:
preprocessor.upload_iteration_metadata()

Scanning for experimenter: closedlooptest
	header_path: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/data_dj_format/20251008/1
		header_name: 20251008_right.ini
		Already present: {'experimenter': 'closedlooptest', 'date': datetime.datetime(2025, 10, 8, 0, 0), 'exp_num': 1}
Found 4 files in 1 fields for key={'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 8), 'exp_num': 1, 'raw_id': 1}
	Skipping field `{'field': 'GCL2', 'region': 'RR', 'cond1': 'iter0', 'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 8), 'exp_num': 1, 'raw_id': 1}` because it already exists


Processes: 100%|██████████| 2/2 [00:00<00:00, 24.25it/s]


In [11]:
missing_keys = dj_table_holder("RoiMask")().list_missing_field()
if len(missing_keys) == 1:
    field_key = missing_keys[0]
    print(f"Missing field key found: {field_key}")
elif len(missing_keys) > 1:
    raise ValueError(f"Multiple missing fields found: {missing_keys}")
else:
    print("No missing fields found, using the last field key.")
    all_field_key = dj_table_holder("Field")().proj().fetch(as_dict=True)
    field_key = all_field_key[-1]
    print(f"Field key: {field_key}")

No missing fields found, using the last field key.
Field key: {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 8), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL2', 'region': 'RR', 'cond1': 'iter0'}


### Visualize cell tpe and quality

In [16]:
from model_in_the_loop.core.gui import ExtendedAutoRoiGui
gui = ExtendedAutoRoiGui(
    dj_table_holder=dj_table_holder, 
    dj_preprocessor=preprocessor,
    all_dj_wrappers=[quality_type_analysis_wrapper, sta_wrapper, random_seed_mei_wrapper],
    take_roi_rgba_from_this_analysis=quality_type_analysis_wrapper.name,
    field_key=field_key,
    canvas_width=30)

Load model weights for cpu from checkpoint /gpfs01/euler/data/Resources/AutoROIs/models/UNET_v0.1.0/dropout_and_aug_regul.ckpt using config /gpfs01/euler/data/Resources/AutoROIs/models/UNET_v0.1.0/sd_images.yaml
Loaded initial ROI mask with 90 ROIs.


In [17]:
display(gui.start_gui())

# The stimulus

### select roi ids and seds

In [ ]:
roi_seed = []
print(len(roi_seed))

In [ ]:
roi_id2mei_id,roi_id2mei_info = random_seed_mei_wrapper.select_subset_of_meis_for_each_roi()

In [ ]:
create_single_mei_avis_and_metadata(
    rois_seed=roi_seed,
    roi_id2mei_ids = roi_id2mei_id,    
    mei_data_container=random_seed_mei_wrapper.mei_data_container,
    stimulus_table=dj_table_holder("Stimulus")(),
    fit_gauss_2d_rf_table= dj_table_holder("FitGauss2DRF")(),
    abs_save_dir=cfg.paths.stimulus_output_dir,
)

# Save data

In [ ]:
# save data 
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = os.path.join(cfg.paths.repo_directory,"data/pipeline_saved_data_during_experiment",timestamp)
random_seed_mei_wrapper.save_all_data_to_dir(save_dir=save_dir)

# Clean up

In [ ]:
userinput = input("Cleanup? (y/n): ")
if userinput.lower() == 'y':
    dj_table_holder.clear_tables("all")
